# Track A v1: Cross-Asset Regime Detection with Gaussian HMM

这个 notebook 是 Track A 的 v1 版本，使用 Yahoo Finance 的市场数据和 FRED 的利率曲线数据，先做一个更完整的跨资产 regime detection 流程。  
This notebook is the v1 version of Track A. It combines Yahoo Finance market data with FRED yield-curve data to build a more complete cross-asset regime detection workflow.

本版本补上三件关键事情：  
This version adds three concrete upgrades:

1. 补入 `HYG/LQD` 信用利差代理变量和 `10Y-2Y` 利率曲线斜率。  
   Add an `HYG/LQD` credit-spread proxy and the `10Y-2Y` yield-curve slope.
2. 把每个 latent state 的市场含义写成中英双语解释。  
   Turn each latent state into a bilingual market interpretation.
3. 给每个 regime 配一个组合规则映射。  
   Map each regime to a portfolio rule.

这里的目标不是直接预测明天涨跌，而是先把市场环境划分成更可解释的 latent regimes。  
The goal is not to forecast tomorrow's return directly, but to partition the market into more interpretable latent regimes.


In [ ]:
import importlib.util

# 只检查依赖，不在 notebook 内自动安装，避免 Windows/Jupyter 子进程编码问题。
# Check dependencies without installing inside the notebook to avoid Windows/Jupyter subprocess encoding issues.
REQUIRED_PACKAGES = ["yfinance", "hmmlearn"]

missing_packages = [pkg for pkg in REQUIRED_PACKAGES if importlib.util.find_spec(pkg) is None]

if missing_packages:
    missing_text = ", ".join(missing_packages)
    raise ImportError(
        f"Missing packages: {missing_text}. 请先在当前环境里安装，例如: pip install yfinance hmmlearn"
    )

print("Dependency check passed / 依赖检查通过: yfinance, hmmlearn")


In [ ]:
import os
import warnings

# Windows + MKL 下 KMeans 初始化容易反复给出线程警告，这里先把线程数限制在 1。
# On Windows + MKL, KMeans initialization can emit repeated thread warnings, so we cap threads at 1.
os.environ.setdefault("OMP_NUM_THREADS", "1")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from hmmlearn.hmm import GaussianHMM
import yfinance as yf

warnings.filterwarnings(
    "ignore",
    message="KMeans is known to have a memory leak on Windows with MKL.*",
    category=UserWarning,
)

plt.style.use("ggplot")

START_DATE = "2015-01-01"
MARKET_TICKERS = {
    "SPY": "US equities / 美国股票",
    "TLT": "Long Treasuries / 长久期美债",
    "GLD": "Gold / 黄金",
    "UUP": "US dollar ETF proxy / 美元 ETF 代理",
    "HYG": "High yield credit / 高收益信用",
    "LQD": "Investment-grade credit / 投资级信用",
    "^VIX": "Volatility index / 波动率指数",
}
FRED_SERIES = {
    "DGS10": "10-Year Treasury yield / 10 年期美债收益率",
    "DGS2": "2-Year Treasury yield / 2 年期美债收益率",
}
REGIME_NAME_CN = {
    "stress": "压力",
    "transition": "过渡",
    "recovery": "修复",
    "risk-on": "风险偏好",
    "risk-off": "风险规避",
}
PORTFOLIO_RULE_LIBRARY = {
    "risk-on": {
        "rule_cn": "风险平价基准",
        "rule_en": "Risk parity core",
        "tilt_cn": "股票、债券、黄金相对均衡，保留少量美元/现金缓冲。",
        "tilt_en": "Keep a balanced mix of equities, bonds, and gold, with a small dollar/cash buffer.",
        "example_weights": "SPY 35% / TLT 30% / GLD 15% / HYG 10% / UUP-cash 10%",
    },
    "recovery": {
        "rule_cn": "偏进攻的最大夏普倾向",
        "rule_en": "Max-Sharpe tilt",
        "tilt_cn": "提高股票和信用权重，保留部分债券与黄金做缓冲。",
        "tilt_en": "Lean into equities and credit while keeping some bonds and gold as ballast.",
        "example_weights": "SPY 40% / HYG 20% / TLT 20% / GLD 10% / UUP-cash 10%",
    },
    "transition": {
        "rule_cn": "均衡防守",
        "rule_en": "Balanced defensive mix",
        "tilt_cn": "降低股票 beta，债券、黄金和现金比重更高。",
        "tilt_en": "Lower equity beta and raise the share of bonds, gold, and cash.",
        "example_weights": "SPY 20% / TLT 35% / GLD 20% / LQD 10% / UUP-cash 15%",
    },
    "stress": {
        "rule_cn": "债券 + 黄金 + 现金",
        "rule_en": "Bonds + gold + cash",
        "tilt_cn": "显著削减股票和高收益信用，优先防御资产。",
        "tilt_en": "Cut equities and high-yield credit aggressively and emphasize defensive assets.",
        "example_weights": "TLT 45% / GLD 25% / LQD 10% / UUP-cash 20%",
    },
    "risk-off": {
        "rule_cn": "债券 + 黄金 + 现金",
        "rule_en": "Bonds + gold + cash",
        "tilt_cn": "整体偏防御，避免高 beta 风险敞口。",
        "tilt_en": "Stay defensive overall and avoid high-beta exposures.",
        "example_weights": "TLT 40% / GLD 20% / LQD 15% / UUP-cash 25%",
    },
}
STATE_CANDIDATES = [2, 3, 4]
N_RESTARTS = 8
RANDOM_STATE = 42


In [ ]:
def load_fred_series(series_id: str) -> pd.DataFrame:
    # FRED 可以直接通过 CSV 拉取，这样 notebook 不需要额外依赖。
    # FRED can be pulled as a simple CSV, which keeps the notebook dependency-light.
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
    frame = pd.read_csv(url)

    date_col = next((col for col in ["DATE", "date", "observation_date"] if col in frame.columns), None)
    if date_col is None:
        raise ValueError(
            f"Could not find a date column in FRED response for {series_id}. Columns: {list(frame.columns)}"
        )

    value_cols = [col for col in frame.columns if col != date_col]
    if not value_cols:
        raise ValueError(
            f"Could not find a value column in FRED response for {series_id}. Columns: {list(frame.columns)}"
        )

    value_col = value_cols[0]
    frame[date_col] = pd.to_datetime(frame[date_col])
    frame[value_col] = pd.to_numeric(frame[value_col], errors="coerce")
    return frame[[date_col, value_col]].rename(columns={date_col: "date", value_col: series_id}).set_index("date")


def download_macro_panel(target_index: pd.Index) -> pd.DataFrame:
    # 把 FRED 日频利率数据对齐到市场交易日，并对缺口做前向填充。
    # Align FRED daily rates to market trading days and forward-fill across gaps.
    fred_df = pd.concat([load_fred_series(series_id) for series_id in FRED_SERIES], axis=1).sort_index()
    fred_df = fred_df.reindex(target_index).ffill()
    fred_df["curve_slope_10y2y"] = fred_df["DGS10"] - fred_df["DGS2"]
    return fred_df


def build_features(close: pd.DataFrame, macro: pd.DataFrame):
    # 这里把最小市场特征集扩展到 v1：加入信用代理和利率曲线斜率。
    # Extend the minimal feature set to v1 by adding a credit proxy and the curve slope.
    close = close.rename(columns={"^VIX": "VIX"}).dropna(how="all")
    returns = close.pct_change()
    credit_spread_proxy = np.log(close["HYG"] / close["LQD"])

    features = pd.DataFrame(
        {
            "SPY_ret": returns["SPY"],
            "SPY_vol_21d": returns["SPY"].rolling(21).std() * np.sqrt(252),
            "VIX_level": close["VIX"],
            "TLT_ret": returns["TLT"],
            "GLD_ret": returns["GLD"],
            "UUP_ret": returns["UUP"],
            "credit_spread_proxy": credit_spread_proxy,
            "curve_slope_10y2y": macro["curve_slope_10y2y"],
            "corr_60d_SPY_TLT": returns["SPY"].rolling(60).corr(returns["TLT"]),
        }
    )

    return close, returns, features.dropna()


def hmm_parameter_count(n_states: int, n_features: int, covariance_type: str = "diag") -> int:
    startprob = n_states - 1
    transmat = n_states * (n_states - 1)
    means = n_states * n_features

    if covariance_type == "diag":
        covars = n_states * n_features
    elif covariance_type == "full":
        covars = n_states * n_features * (n_features + 1) // 2
    else:
        raise ValueError(f"Unsupported covariance_type: {covariance_type}")

    return startprob + transmat + means + covars


def fit_best_hmm(
    X: np.ndarray,
    n_states: int,
    covariance_type: str = "diag",
    n_restarts: int = 8,
    n_iter: int = 500,
    random_state: int = 42,
):
    # 多次随机重启，避免 HMM 卡在较差的局部最优。
    # Use multiple random restarts so the HMM is less likely to get stuck in a poor local optimum.
    best_model = None
    best_score = -np.inf

    for seed in range(random_state, random_state + n_restarts):
        model = GaussianHMM(
            n_components=n_states,
            covariance_type=covariance_type,
            n_iter=n_iter,
            random_state=seed,
        )
        model.fit(X)
        score = model.score(X)
        if score > best_score:
            best_score = score
            best_model = model

    return best_model, best_score


def select_hmm_by_bic(
    X: np.ndarray,
    state_candidates,
    covariance_type: str = "diag",
    n_restarts: int = 8,
    random_state: int = 42,
):
    rows = []
    models = {}
    n_features = X.shape[1]
    n_obs = X.shape[0]

    for n_states in state_candidates:
        model, loglik = fit_best_hmm(
            X=X,
            n_states=n_states,
            covariance_type=covariance_type,
            n_restarts=n_restarts,
            random_state=random_state,
        )
        n_params = hmm_parameter_count(n_states, n_features, covariance_type=covariance_type)
        bic = -2 * loglik + n_params * np.log(n_obs)
        rows.append({"n_states": n_states, "loglik": loglik, "n_params": n_params, "bic": bic})
        models[n_states] = model

    summary = pd.DataFrame(rows).sort_values("n_states").reset_index(drop=True)
    best_n_states = int(summary.sort_values("bic").iloc[0]["n_states"])
    return summary, models, best_n_states


def label_states(feature_frame: pd.DataFrame, state_col: str = "state"):
    # 风险分数越高，越偏向 risk-on；越低，越偏向 stress / risk-off。
    # Higher risk scores imply more risk-on behavior; lower scores imply stress / risk-off behavior.
    metric_cols = [
        "SPY_ret",
        "SPY_vol_21d",
        "VIX_level",
        "credit_spread_proxy",
        "curve_slope_10y2y",
        "corr_60d_SPY_TLT",
    ]
    state_means = feature_frame.groupby(state_col)[metric_cols].mean()

    state_stds = state_means.std(ddof=0).replace(0, 1)
    z = (state_means - state_means.mean()) / state_stds
    risk_score = (
        z["SPY_ret"].fillna(0)
        - z["SPY_vol_21d"].fillna(0)
        - z["VIX_level"].fillna(0)
        + z["credit_spread_proxy"].fillna(0)
        + 0.75 * z["curve_slope_10y2y"].fillna(0)
        - 0.5 * z["corr_60d_SPY_TLT"].fillna(0)
    )

    ordered_states = risk_score.sort_values().index.tolist()
    n_states = len(ordered_states)

    if n_states == 2:
        names_en = ["risk-off", "risk-on"]
    elif n_states == 3:
        names_en = ["stress", "transition", "risk-on"]
    elif n_states == 4:
        names_en = ["stress", "transition", "recovery", "risk-on"]
    else:
        names_en = [f"state_{i}" for i in range(n_states)]

    label_map_en = {state: names_en[i] for i, state in enumerate(ordered_states)}
    label_map_cn = {state: REGIME_NAME_CN.get(label_map_en[state], label_map_en[state]) for state in ordered_states}
    return label_map_en, label_map_cn, state_means, risk_score.sort_values(), ordered_states


def build_state_interpretation(state_summary: pd.DataFrame, overall_means: pd.Series) -> pd.DataFrame:
    rows = []

    for _, row in state_summary.iterrows():
        cn_parts = []
        en_parts = []

        if row["avg_SPY_ret"] >= overall_means["SPY_ret"]:
            cn_parts.append("股票回报相对更强")
            en_parts.append("equity returns are stronger")
        else:
            cn_parts.append("股票回报相对更弱")
            en_parts.append("equity returns are weaker")

        if row["avg_SPY_vol_21d"] >= overall_means["SPY_vol_21d"]:
            cn_parts.append("波动率偏高")
            en_parts.append("volatility is elevated")
        else:
            cn_parts.append("波动率偏低")
            en_parts.append("volatility is subdued")

        if row["avg_VIX_level"] >= overall_means["VIX_level"]:
            cn_parts.append("VIX 处在更高位置")
            en_parts.append("the VIX sits at a higher level")
        else:
            cn_parts.append("VIX 更平稳")
            en_parts.append("the VIX is calmer")

        if row["avg_credit_spread_proxy"] >= overall_means["credit_spread_proxy"]:
            cn_parts.append("信用风险偏好更强")
            en_parts.append("credit appetite is stronger")
        else:
            cn_parts.append("信用环境更偏防守")
            en_parts.append("credit conditions are more defensive")

        if row["avg_curve_slope_10y2y"] >= overall_means["curve_slope_10y2y"]:
            cn_parts.append("利率曲线更陡")
            en_parts.append("the yield curve is steeper")
        else:
            cn_parts.append("利率曲线更平或更倒挂")
            en_parts.append("the yield curve is flatter or more inverted")

        if row["avg_corr_60d_SPY_TLT"] >= overall_means["corr_60d_SPY_TLT"]:
            cn_parts.append("股债分散效果较弱")
            en_parts.append("stock-bond diversification is weaker")
        else:
            cn_parts.append("股债分散效果较强")
            en_parts.append("stock-bond diversification is stronger")

        rows.append(
            {
                "state": row["state"],
                "regime_cn": row["regime_cn"],
                "regime_en": row["regime_en"],
                "interpretation_cn": "；".join(cn_parts) + "。",
                "interpretation_en": "; ".join(en_parts).capitalize() + ".",
            }
        )

    return pd.DataFrame(rows)


def build_portfolio_mapping(state_summary: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for _, row in state_summary.iterrows():
        rule = PORTFOLIO_RULE_LIBRARY[row["regime_en"]]
        rows.append(
            {
                "state": row["state"],
                "regime_cn": row["regime_cn"],
                "regime_en": row["regime_en"],
                "portfolio_rule_cn": rule["rule_cn"],
                "portfolio_rule_en": rule["rule_en"],
                "allocation_tilt_cn": rule["tilt_cn"],
                "allocation_tilt_en": rule["tilt_en"],
                "example_weights": rule["example_weights"],
            }
        )

    return pd.DataFrame(rows)


def shade_states(ax, index, state_series, colors, alpha: float = 0.15):
    start = 0
    index = pd.Index(index)

    while start < len(state_series):
        state = state_series.iloc[start]
        end = start
        while end + 1 < len(state_series) and state_series.iloc[end + 1] == state:
            end += 1

        ax.axvspan(index[start], index[end], color=colors[state], alpha=alpha, lw=0)
        start = end + 1


In [ ]:
# 从 Yahoo 下载市场资产数据，再把 FRED 曲线数据对齐到交易日。
# Download market assets from Yahoo, then align FRED curve data to trading days.
raw = yf.download(
    tickers=list(MARKET_TICKERS),
    start=START_DATE,
    auto_adjust=True,
    progress=False,
)

close = raw["Close"].copy()
macro = download_macro_panel(close.index)
close, returns, features = build_features(close, macro)

display(features.head())
display(features.describe().T[["mean", "std", "min", "max"]])
display(macro[["DGS10", "DGS2", "curve_slope_10y2y"]].tail())

print("Feature sample starts / 特征样本起点:", features.index.min().date())
print("Feature sample ends / 特征样本终点:", features.index.max().date())
print("Observations / 样本数:", len(features))


In [ ]:
# 标准化特征后，用 BIC 在 2/3/4 个状态之间选择最合适的 Gaussian HMM。
# Standardize the features and use BIC to choose the best Gaussian HMM among 2/3/4 states.
scaler = StandardScaler()
X = scaler.fit_transform(features)

bic_summary, model_store, best_n_states = select_hmm_by_bic(
    X=X,
    state_candidates=STATE_CANDIDATES,
    covariance_type="diag",
    n_restarts=N_RESTARTS,
    random_state=RANDOM_STATE,
)

best_model = model_store[best_n_states]
hidden_states = pd.Series(best_model.predict(X), index=features.index, name="state")

analysis_df = features.copy()
analysis_df["state"] = hidden_states

label_map_en, label_map_cn, raw_state_means, state_risk_score, ordered_states = label_states(analysis_df)
analysis_df["regime_en"] = analysis_df["state"].map(label_map_en)
analysis_df["regime_cn"] = analysis_df["state"].map(label_map_cn)

state_summary = (
    analysis_df.groupby("state")
    .agg(
        n_days=("SPY_ret", "size"),
        avg_SPY_ret=("SPY_ret", "mean"),
        avg_SPY_vol_21d=("SPY_vol_21d", "mean"),
        avg_VIX_level=("VIX_level", "mean"),
        avg_TLT_ret=("TLT_ret", "mean"),
        avg_GLD_ret=("GLD_ret", "mean"),
        avg_UUP_ret=("UUP_ret", "mean"),
        avg_credit_spread_proxy=("credit_spread_proxy", "mean"),
        avg_curve_slope_10y2y=("curve_slope_10y2y", "mean"),
        avg_corr_60d_SPY_TLT=("corr_60d_SPY_TLT", "mean"),
    )
    .loc[ordered_states]
    .reset_index()
)
state_summary["regime_en"] = state_summary["state"].map(label_map_en)
state_summary["regime_cn"] = state_summary["state"].map(label_map_cn)
state_summary["share"] = state_summary["n_days"] / len(analysis_df)
state_summary = state_summary[
    [
        "state",
        "regime_cn",
        "regime_en",
        "n_days",
        "share",
        "avg_SPY_ret",
        "avg_SPY_vol_21d",
        "avg_VIX_level",
        "avg_TLT_ret",
        "avg_GLD_ret",
        "avg_UUP_ret",
        "avg_credit_spread_proxy",
        "avg_curve_slope_10y2y",
        "avg_corr_60d_SPY_TLT",
    ]
]

overall_means = analysis_df[
    [
        "SPY_ret",
        "SPY_vol_21d",
        "VIX_level",
        "credit_spread_proxy",
        "curve_slope_10y2y",
        "corr_60d_SPY_TLT",
    ]
].mean()

regime_interpretation = build_state_interpretation(state_summary, overall_means)
portfolio_mapping = build_portfolio_mapping(state_summary)

transition_matrix = pd.DataFrame(
    best_model.transmat_,
    index=[f"state_{state} ({label_map_cn[state]} / {label_map_en[state]})" for state in range(best_n_states)],
    columns=[f"state_{state} ({label_map_cn[state]} / {label_map_en[state]})" for state in range(best_n_states)],
)

print("BIC summary / BIC 汇总")
display(bic_summary.round(2))

print("State summary / 状态统计")
display(state_summary.round(4))

print("State interpretation / 状态解释")
display(regime_interpretation)

print("Portfolio mapping / 组合规则映射")
display(portfolio_mapping)

print("Transition matrix / 转移矩阵")
display(transition_matrix.round(3))

print(f"Selected number of states by BIC / BIC 选择的状态数: {best_n_states}")


## 图怎么看 / How to read the charts

- 第一张图只比较不同 hidden state 个数的 BIC，数值越低通常越好。  
  The first chart only compares BIC across different hidden-state counts; lower is usually better.
- 彩色背景表示 HMM 推断出来的 regime 区间。  
  The colored background marks the regimes inferred by the HMM.
- 如果背景切换得很碎，往往不是模型直接坏了，而是说明特征还可以继续增强。  
  If the background switches too often, the model is not necessarily broken; it often means the feature set can still be improved.


In [ ]:
# 把市场价格、波动、相关性、信用代理和利率曲线一起画出来，方便看 regime 的直觉含义。
# Plot prices, volatility, correlation, credit proxy, and curve slope together so the regimes are easier to interpret.
aligned_close = close.rename(columns={"^VIX": "VIX"}).loc[analysis_df.index]
aligned_macro = macro.loc[analysis_df.index]

ordered_palette = ["#b2182b", "#fdae61", "#67a9cf", "#1a9850", "#762a83"]
state_colors = {state: ordered_palette[i] for i, state in enumerate(ordered_states)}
legend_handles = [
    Patch(facecolor=state_colors[state], alpha=0.25, label=f"state {state}: {label_map_cn[state]} / {label_map_en[state]}")
    for state in ordered_states
]

fig_bic, ax_bic = plt.subplots(figsize=(10, 4))
ax_bic.plot(bic_summary["n_states"], bic_summary["bic"], marker="o", linewidth=1.5)
ax_bic.axvline(best_n_states, color="black", linestyle="--", alpha=0.7)
ax_bic.set_xticks(bic_summary["n_states"])
ax_bic.set_title("BIC across Gaussian HMM state counts / 隐状态数量的 BIC 比较")
ax_bic.set_xlabel("Number of hidden states / 隐状态数量")
ax_bic.set_ylabel("BIC")
fig_bic.tight_layout()
plt.show()

fig_market, axes = plt.subplots(
    3,
    1,
    figsize=(14, 14),
    sharex=True,
    gridspec_kw={"height_ratios": [2, 1.5, 1.5]},
)

axes[0].plot(aligned_close.index, aligned_close["SPY"], color="black", linewidth=1.5)
shade_states(axes[0], aligned_close.index, analysis_df["state"], state_colors)
axes[0].set_title("SPY with inferred regimes / SPY 价格与 regime")
axes[0].set_ylabel("Adjusted close / 复权收盘价")

axes[1].plot(analysis_df.index, analysis_df["VIX_level"], color="#4c72b0", linewidth=1.3)
shade_states(axes[1], analysis_df.index, analysis_df["state"], state_colors)
axes[1].set_title("VIX level / VIX 水平")
axes[1].set_ylabel("Index level / 指数水平")

axes[2].plot(analysis_df.index, analysis_df["corr_60d_SPY_TLT"], color="#55a868", linewidth=1.3)
shade_states(axes[2], analysis_df.index, analysis_df["state"], state_colors)
axes[2].axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.6)
axes[2].set_title("60-day rolling stock-bond correlation / 60 日股债滚动相关性")
axes[2].set_ylabel("Correlation / 相关系数")
axes[2].legend(handles=legend_handles, loc="upper left", ncol=min(2, len(legend_handles)))

fig_market.tight_layout()
plt.show()

fig_macro, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

axes[0].plot(analysis_df.index, analysis_df["credit_spread_proxy"], color="#8172b2", linewidth=1.3)
shade_states(axes[0], analysis_df.index, analysis_df["state"], state_colors)
axes[0].set_title("HYG/LQD credit proxy / HYG-LQD 信用代理")
axes[0].set_ylabel("Log ratio / 对数比值")

axes[1].plot(aligned_macro.index, aligned_macro["curve_slope_10y2y"], color="#c44e52", linewidth=1.3)
shade_states(axes[1], analysis_df.index, analysis_df["state"], state_colors)
axes[1].axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.6)
axes[1].set_title("10Y - 2Y slope / 10 年期减 2 年期利差")
axes[1].set_ylabel("Percent / 百分点")

fig_macro.tight_layout()
plt.show()


In [ ]:
# 用热力图看各个 state 在不同特征上的相对高低，颜色是按列做的 z-score。
# Use a heatmap to compare states across features; colors are column-wise z-scores.
ordered_feature_means = analysis_df.groupby("state")[
    [
        "SPY_ret",
        "SPY_vol_21d",
        "VIX_level",
        "TLT_ret",
        "GLD_ret",
        "UUP_ret",
        "credit_spread_proxy",
        "curve_slope_10y2y",
        "corr_60d_SPY_TLT",
    ]
].mean().loc[ordered_states]


def zscore_column(col: pd.Series) -> pd.Series:
    std = col.std(ddof=0)
    if std == 0 or pd.isna(std):
        return pd.Series(np.zeros(len(col)), index=col.index)
    return (col - col.mean()) / std


heat_values = ordered_feature_means.apply(zscore_column, axis=0)

fig, ax = plt.subplots(figsize=(12, 4.2 + 0.45 * len(ordered_states)))
im = ax.imshow(heat_values.values, cmap="RdYlGn", aspect="auto", vmin=-2, vmax=2)

ax.set_xticks(range(len(heat_values.columns)))
ax.set_xticklabels(heat_values.columns, rotation=45, ha="right")
ax.set_yticks(range(len(ordered_states)))
ax.set_yticklabels([f"state {state}: {label_map_cn[state]} / {label_map_en[state]}" for state in ordered_states])
ax.set_title("State feature means with column-wise z-score colors / 各状态特征均值热力图")

for i in range(ordered_feature_means.shape[0]):
    for j in range(ordered_feature_means.shape[1]):
        ax.text(
            j,
            i,
            f"{ordered_feature_means.iloc[i, j]:.3f}",
            ha="center",
            va="center",
            fontsize=8,
        )

fig.colorbar(im, ax=ax, label="z-score within feature / 特征内 z-score")
plt.tight_layout()
plt.show()


## Next Steps / 下一步

这一版已经把 Track A 的 v1 骨架搭起来了：有更完整的特征集、状态解释和组合映射。  
This version already gives Track A a v1 structure: a fuller feature set, regime interpretation, and portfolio mapping.

如果要继续向原始 slide 靠拢，最自然的下一步是：  
If you want to move closer to the original slide, the next logical upgrades are:

- 加入 oil 和 VIX term structure。  
  Add oil and a VIX term-structure feature.
- 在 portfolio mapping 基础上做一个简单 backtest。  
  Build a simple backtest on top of the portfolio mapping.
- 比较 HMM labels 和后续 Track B embedding clusters 的一致性。  
  Compare the HMM labels with the later Track B embedding clusters.
